# Regresión No Lineal (Análisis Predictivo - Socio Formador)
## Base de datos: `Ventas_limpio.csv` (la misma de la Actividad 3, Regresión Lineal)

**Objetivo.** Elegir 2 modelos de correlación no lineal y analizar la correlación que existe entre las variables numéricas seleccionadas, aplicando la herramienta de **Regresión No Lineal** (`curve_fit`).

**Rodrigo Ocampo | A01752337**

In [ ]:
#Cargamos librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
#Carga desde un archivo .csv
ventas = pd.read_csv('Ventas_limpio.csv')
ventas.head()

## Preparación de los datos (igual que en la regresión lineal)

Se usan las **3 variables numéricas** de la base: `total_unidades`, `precio_unitario` y `total_neto`. Los identificadores (folio, cp, articulo_id) no se usan porque sus números no representan una cantidad.

Igual que en la Actividad 3, las variables se pasan a **rangos ordenados**, porque hay valores extremos (un escáner de más de \$2 millones contra dientes de \$170) que dominarían cualquier curva:

- **Unidades:** Unitaria (1), Par (2), Pocas (3-5), Mayoreo (6-250)
- **Precio unitario (quintiles):** Muy económico, Económico, Medio, Alto, Premium
- **Venta total (quintiles):** Mínima, Baja, Media, Alta, Top

*Año y mes no se incluyen porque en la regresión lineal no explicaron nada (R² ≈ 0.02).*

In [ ]:
#Categorizamos con cut y qcut (mismos rangos que en la regresion lineal)
df = ventas.copy()

df['nuevo_unidades'] = pd.cut(
    df['total_unidades'], bins=[0, 1, 2, 5, 250],
    labels=['Unitaria', 'Par', 'Pocas', 'Mayoreo']
)

df['nuevo_precio_unit'] = pd.qcut(
    df['precio_unitario'], q=5,
    labels=['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium'],
    duplicates='drop'
)

df['nuevo_total'] = pd.qcut(
    df['total_neto'], q=5,
    labels=['Mínima', 'Baja', 'Media', 'Alta', 'Top'],
    duplicates='drop'
)

df = df[['nuevo_unidades', 'nuevo_precio_unit', 'nuevo_total']]
df.head()

In [ ]:
#Codificamos con OrdinalEncoder respetando el orden de negocio
categoricas = df.copy()

columnas = ['nuevo_unidades', 'nuevo_precio_unit', 'nuevo_total']

encoder = OrdinalEncoder(
    categories=[
        ['Unitaria', 'Par', 'Pocas', 'Mayoreo'],
        ['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium'],
        ['Mínima', 'Baja', 'Media', 'Alta', 'Top']
    ]
)

categoricas[[col + '_ord' for col in columnas]] = encoder.fit_transform(categoricas[columnas])

columnas_ord = [col + '_ord' for col in columnas]
categoricas[columnas_ord] = categoricas[columnas_ord].astype('int64')

categoricas.drop(columns=columnas, inplace=True)
categoricas.head()

## Correlación entre las variables numéricas

In [ ]:
#Matriz de correlacion y heatmap
correlacion = categoricas.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de correlación de variables de ventas')
plt.show()

In [ ]:
#Graficamos todas las dispersiones entre todas las variables
#(se usa una muestra de 3,000 ventas para que la grafica no tarde tanto)
sns.pairplot(categoricas.sample(3000, random_state=0))
plt.show()

## Elección de las variables

La pareja con mayor correlación es **precio unitario (X)** y **venta total (Y)**, con **r = 0.77**. Es la misma pareja de la regresión lineal simple, así que podemos comparar si una curva la explica mejor que una recta.

Antes de elegir los modelos, revisamos el **promedio de Y en cada nivel de X**. Si la relación fuera lineal, el promedio subiría siempre lo mismo de un nivel a otro.

In [ ]:
#Promedio de la venta total por cada rango de precio
tabla_niveles = pd.DataFrame({
    'Promedio venta total (Y)': categoricas.groupby('nuevo_precio_unit_ord')['nuevo_total_ord'].mean()
})
tabla_niveles['Diferencia vs nivel anterior'] = tabla_niveles['Promedio venta total (Y)'].diff()
tabla_niveles.index = ['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium']
round(tabla_niveles, 2)

**Lectura:** el promedio **no sube de forma pareja**. Baja de "Muy económico" a "Económico" (−0.33) y luego sube más fuerte (+1.11 y +1.24). Esto pasa porque los productos más baratos (fresas, dientes) se compran por volumen y su total termina siendo más alto que el de los "Económicos", que se venden sueltos. Una recta no puede seguir ese "bache"; una curva sí. Por eso tiene sentido probar una **regresión no lineal**.

In [ ]:
#Declaramos las variables dependientes e independientes para la regresión No lineal
Vars_Indep = categoricas[['nuevo_precio_unit_ord']]
Var_Dep = categoricas['nuevo_total_ord']

In [ ]:
#Redefinimos las variables
x = Vars_Indep
y = Var_Dep

**Modelos elegidos (de la lista vista en clase):**

            y = a*x^2 + b*x + c      ("Función cuadrática")

            y = a*exp(-b*x) + c      ("Función exponencial")

Como punto de referencia, primero calculamos la **regresión lineal** (Y = b0 + b1·X) que ya teníamos en la Actividad 3.

**MODELO DE REFERENCIA: LINEAL**

In [ ]:
#Regresion lineal (la misma de la Actividad 3) para comparar
modelo_lineal = LinearRegression()
modelo_lineal.fit(x, y)
y_pred_lineal = modelo_lineal.predict(x)

R2_Lineal = r2_score(y, y_pred_lineal)
print('Ecuación lineal: y =', round(modelo_lineal.intercept_, 4), '+', round(modelo_lineal.coef_[0], 4), '* x')
print('R2 lineal:', round(R2_Lineal, 4))

**MODELO 1: FUNCIÓN CUADRÁTICA**

In [ ]:
def func1(x, a, b, c):
    return a*x**2 + b*x + c

In [ ]:
#Ajustamos los parámetros de la función curve_fit
parametros1, covs1 = curve_fit(func1, categoricas['nuevo_precio_unit_ord'], categoricas['nuevo_total_ord'])

In [ ]:
#Obtenemos los coeficientes del modelo de regresión no lineal
parametros1

In [ ]:
# Predicciones
y_pred1 = func1(x, *parametros1)
y_pred1

In [ ]:
#Ecuacion del modelo
print('y =', round(parametros1[0], 4), '* x^2 +', round(parametros1[1], 4), '* x +', round(parametros1[2], 4))

In [ ]:
#Graficamos las predicciones y los datos originales para realizar la comparación
#(jitter: movemos un poco los puntos para que no se encimen, porque X y Y solo valen 0 a 4)
np.random.seed(0)
x_jitter = categoricas['nuevo_precio_unit_ord'] + np.random.uniform(-0.2, 0.2, len(categoricas))
y_jitter = categoricas['nuevo_total_ord'] + np.random.uniform(-0.2, 0.2, len(categoricas))

x_curva = np.linspace(0, 4, 100)

plt.figure(figsize=(9, 6))
plt.plot(x_jitter, y_jitter, 'o', color='blue', alpha=0.03, markersize=3, label='y-original')
plt.plot(x_curva, func1(x_curva, *parametros1), color='red', linewidth=3, label='y-predecida con y=a*x^2+b*x+c')
plt.plot([0, 1, 2, 3, 4], tabla_niveles['Promedio venta total (Y)'], 's', color='black', markersize=9, label='Promedio real por nivel')
plt.xticks([0, 1, 2, 3, 4], ['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium'])
plt.yticks([0, 1, 2, 3, 4], ['Mínima', 'Baja', 'Media', 'Alta', 'Top'])
plt.xlabel('Precio unitario (rango)')
plt.ylabel('Venta total (rango)')
plt.title('Modelo 1: Regresión cuadrática')
plt.legend(loc='best', fancybox=True, shadow=True)
plt.grid(True)
plt.show()

In [ ]:
#Calculamos el coeficiente de determinación del modelo
R2_Modelo1 = r2_score(y, y_pred1)
R2_Modelo1

In [ ]:
#Calculamos el coeficiente de correlación del modelo
R1 = np.sqrt(R2_Modelo1)
R1

**MODELO 2: FUNCIÓN EXPONENCIAL**

*Nota:* `curve_fit` busca los parámetros probando valores, y si no le decimos dónde empezar arranca con a = 1, b = 1, c = 1. Con la exponencial eso hace que se "atore" y termine en algo casi igual a una recta. Por eso le damos un punto de partida con `p0`: un valor de **b negativo** indica una curva que **crece** (porque −b·x se vuelve positivo).

In [ ]:
def func2(x, a, b, c):
    return a*np.exp(-b*x) + c

In [ ]:
#Ajustamos los parámetros de la función curve_fit (con punto de partida p0)
parametros2, covs2 = curve_fit(func2, categoricas['nuevo_precio_unit_ord'], categoricas['nuevo_total_ord'], p0=[1, -0.3, 0])

In [ ]:
#Obtenemos los coeficientes del modelo de regresión no lineal
parametros2

In [ ]:
# Predicciones
y_pred2 = func2(x, *parametros2)
y_pred2

In [ ]:
#Ecuacion del modelo
print('y =', round(parametros2[0], 4), '* exp(-(', round(parametros2[1], 4), ') * x) +', round(parametros2[2], 4))

In [ ]:
#Graficamos las predicciones y los datos originales para realizar la comparación
plt.figure(figsize=(9, 6))
plt.plot(x_jitter, y_jitter, 'o', color='blue', alpha=0.03, markersize=3, label='y-original')
plt.plot(x_curva, func2(x_curva, *parametros2), color='green', linewidth=3, label='y-predecida con y=a*exp(-b*x)+c')
plt.plot([0, 1, 2, 3, 4], tabla_niveles['Promedio venta total (Y)'], 's', color='black', markersize=9, label='Promedio real por nivel')
plt.xticks([0, 1, 2, 3, 4], ['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium'])
plt.yticks([0, 1, 2, 3, 4], ['Mínima', 'Baja', 'Media', 'Alta', 'Top'])
plt.xlabel('Precio unitario (rango)')
plt.ylabel('Venta total (rango)')
plt.title('Modelo 2: Regresión exponencial')
plt.legend(loc='best', fancybox=True, shadow=True)
plt.grid(True)
plt.show()

In [ ]:
#Calculamos el coeficiente de determinación del modelo
R2_Modelo2 = r2_score(y, y_pred2)
R2_Modelo2

In [ ]:
#Calculamos el coeficiente de correlación del modelo
R2 = np.sqrt(R2_Modelo2)
R2

## Comparación de modelos

In [ ]:
#Tabla comparativa
comparacion = pd.DataFrame({
    'Modelo': ['Lineal (referencia)', 'Cuadrático', 'Exponencial'],
    'Ecuación': ['y = b0 + b1*x', 'y = a*x^2 + b*x + c', 'y = a*exp(-b*x) + c'],
    'R2': [R2_Lineal, R2_Modelo1, R2_Modelo2],
    'R': [np.sqrt(R2_Lineal), R1, R2]
})
comparacion['Mejora R2 vs lineal'] = comparacion['R2'] - R2_Lineal
round(comparacion, 4)

In [ ]:
#Grafica de los 3 modelos juntos
plt.figure(figsize=(9, 6))
plt.plot([0, 1, 2, 3, 4], tabla_niveles['Promedio venta total (Y)'], 's', color='black', markersize=9, label='Promedio real por nivel')
plt.plot(x_curva, modelo_lineal.intercept_ + modelo_lineal.coef_[0] * x_curva, '--', color='gray', linewidth=2, label='Lineal (R² = ' + str(round(R2_Lineal, 3)) + ')')
plt.plot(x_curva, func1(x_curva, *parametros1), color='red', linewidth=2, label='Cuadrático (R² = ' + str(round(R2_Modelo1, 3)) + ')')
plt.plot(x_curva, func2(x_curva, *parametros2), color='green', linewidth=2, label='Exponencial (R² = ' + str(round(R2_Modelo2, 3)) + ')')
plt.xticks([0, 1, 2, 3, 4], ['Muy económico', 'Económico', 'Medio', 'Alto', 'Premium'])
plt.yticks([0, 1, 2, 3, 4], ['Mínima', 'Baja', 'Media', 'Alta', 'Top'])
plt.xlabel('Precio unitario (rango)')
plt.ylabel('Venta total (rango)')
plt.title('Comparación: lineal vs cuadrático vs exponencial')
plt.legend(loc='best', fancybox=True, shadow=True)
plt.grid(True)
plt.show()

# Tabla de coeficientes de todas las correlaciones analizadas

Ahora aplicamos los **mismos 3 modelos** (lineal, cuadrático y exponencial) a **todas las parejas** de variables numéricas. Con 3 variables hay 6 combinaciones X → Y (el orden importa: no es lo mismo predecir el total con el precio que el precio con el total).

Para cada pareja se obtiene:
- **r**: correlación de Pearson del heatmap (con signo: + suben juntas, − una sube y la otra baja).
- **R²** de cada modelo: qué % de Y explica.
- **R = √R²**: coeficiente de correlación del modelo (como en clase). En los modelos no lineales R no tiene signo.

*Nota sobre `p0`:* cada pareja necesita su propio punto de partida para la exponencial. Si la relación sube, sirve `[1, 0.3, 0]`; en unidades → precio, que baja cada vez más rápido, se usa `[-1, -0.3, 3]`. Si no se le da, `curve_fit` termina en una recta.

In [ ]:
#Lista de parejas: (X, Y, punto de partida p0 de la exponencial)
parejas = [
    ('nuevo_precio_unit_ord', 'nuevo_total_ord', [1, 0.3, 0]),
    ('nuevo_unidades_ord', 'nuevo_total_ord', [1, 0.3, 0]),
    ('nuevo_total_ord', 'nuevo_precio_unit_ord', [1, 0.3, 0]),
    ('nuevo_total_ord', 'nuevo_unidades_ord', [1, 0.3, 0]),
    ('nuevo_precio_unit_ord', 'nuevo_unidades_ord', [1, 0.3, 0]),
    ('nuevo_unidades_ord', 'nuevo_precio_unit_ord', [-1, -0.3, 3])
]

#Lista para guardar los resultados
resultados = []

#Para cada pareja se repiten los mismos pasos de arriba
for x_col, y_col, p0 in parejas:
    X_par = categoricas[[x_col]]
    Y_par = categoricas[y_col]

    #Correlacion de Pearson (heatmap)
    r_par = correlacion.loc[x_col, y_col]

    #Modelo lineal
    lineal = LinearRegression()
    lineal.fit(X_par, Y_par)
    R2_lin = r2_score(Y_par, lineal.predict(X_par))

    #Modelo 1: cuadratico
    par1, cov1 = curve_fit(func1, categoricas[x_col], categoricas[y_col])
    R2_cuad = r2_score(Y_par, func1(categoricas[x_col], *par1))

    #Modelo 2: exponencial
    par2, cov2 = curve_fit(func2, categoricas[x_col], categoricas[y_col], p0=p0, maxfev=20000)
    R2_exp = r2_score(Y_par, func2(categoricas[x_col], *par2))

    resultados.append({
        'X': x_col,
        'Y': y_col,
        'r (heatmap)': r_par,
        'R2 lineal': R2_lin,
        'R lineal': np.sqrt(R2_lin),
        'R2 cuadratico': R2_cuad,
        'R cuadratico': np.sqrt(R2_cuad),
        'R2 exponencial': R2_exp,
        'R exponencial': np.sqrt(R2_exp)
    })

In [ ]:
#Tabla final de coeficientes
tabla_coeficientes = pd.DataFrame(resultados)

#Nombres cortos para que se lea mejor
nombres_cortos = {
    'nuevo_precio_unit_ord': 'Precio unitario',
    'nuevo_total_ord': 'Venta total',
    'nuevo_unidades_ord': 'Unidades'
}
tabla_coeficientes['X'] = tabla_coeficientes['X'].replace(nombres_cortos)
tabla_coeficientes['Y'] = tabla_coeficientes['Y'].replace(nombres_cortos)

#Mejor modelo de cada pareja (el de mayor R2)
tabla_coeficientes['Mejor modelo'] = tabla_coeficientes[['R2 lineal', 'R2 cuadratico', 'R2 exponencial']].idxmax(axis=1).str.replace('R2 ', '')

round(tabla_coeficientes, 4)

In [ ]:
#Heatmap de los R2 de cada pareja y modelo
tabla_r2 = tabla_coeficientes[['R2 lineal', 'R2 cuadratico', 'R2 exponencial']].copy()
tabla_r2.index = tabla_coeficientes['X'] + ' → ' + tabla_coeficientes['Y']

plt.figure(figsize=(8, 5))
sns.heatmap(tabla_r2, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1)
plt.title('Coeficiente de determinación (R²) por pareja y modelo')
plt.ylabel('Pareja (X → Y)')
plt.show()

**Lectura de la tabla:**

1. **Solo una pareja tiene una relación fuerte:** precio ↔ venta total (R² ≈ 0.59 a 0.63). Las demás quedan por debajo de 0.12, así que ninguna curva las vuelve útiles.
2. **Los dos modelos no lineales siempre igualan o superan al lineal**, y entre ellos quedan casi empatados (diferencias de 0.0001 en 3 parejas). El cuadrático gana en las dos parejas fuertes (precio ↔ venta total). Donde la relación ya es casi recta (venta total → precio, venta total → unidades), los 3 modelos dan prácticamente el mismo R².
3. **La mayor mejora relativa está en precio → unidades:** el R² pasa de 0.037 (lineal) a 0.097 (exponencial). Sigue siendo débil, pero la curva muestra que el cambio está en un solo punto: los productos *Muy económicos* se compran en más unidades y a partir de ahí las unidades casi no cambian.
4. **El orden importa:** precio → total (0.628) no es igual a total → precio (0.595). En el lineal sí da igual (R² = r²), pero en las curvas no.

## Conclusiones

| Modelo | Ecuación | R² | R |
|---|---|---|---|
| Lineal (referencia) | y = 0.45 + 0.78x | 0.593 | 0.770 |
| **Cuadrático** | **y = 0.16x² + 0.14x + 0.76** | **0.628** | **0.792** |
| Exponencial | y = 1.29·e^(0.31x) − 0.63 | 0.619 | 0.787 |

1. **Los dos modelos no lineales explican mejor que la recta.** El cuadrático explica el 62.8% de la variación de la venta total, contra 59.3% del lineal (+3.4 puntos). El exponencial queda en medio (61.9%).
2. **El mejor modelo es el cuadrático.** Su coeficiente `a` es positivo (0.16), así que la curva "se abre hacia arriba": en los precios bajos la venta total casi no cambia, y en los precios altos sube cada vez más rápido. Es el único de los dos que puede seguir el "bache" entre Muy económico y Económico.
3. **El exponencial dice algo parecido:** b es negativo (−0.31), lo que significa que la venta total crece más rápido conforme el precio sube. Pero una exponencial siempre sube, así que no puede bajar en el nivel Económico; por eso queda debajo del cuadrático.
4. **¿Por qué la relación es curva? (negocio).** Los productos Muy económicos (consumibles como fresas y dientes) se compran por volumen, por eso su total es más alto que el de los Económicos, que se venden sueltos. En el otro extremo, el equipo Premium se vende de uno en uno, pero cada pieza ya cae en el rango Top.
5. **La mejora es pequeña.** El precio por sí solo explica alrededor del 60%; el 40% restante depende sobre todo de las unidades (en la regresión múltiple, precio + unidades llegó a R² = 0.84). Una curva ayuda, pero no sustituye a la variable que falta.

**Limitación:** X y Y son rangos (0 a 4), no pesos, así que el modelo sirve para entender la forma de la relación, no para pronosticar montos exactos.